# Real Qwen GPU profile

Loads the pinned pretrained checkpoint from the local project cache and profiles inference batches plus finite-gradient forward/backward on fit-only records. No optimizer steps occur. This is a compute profile, not a validation-quality measurement. Download the checkpoint before running offline. Small-sample timing excludes production disk lookup and does not predict sustained laptop thermals.

In [ ]:
from pathlib import Path
import json
import time
_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else next(
    p/"anshp" for p in (Path.cwd(),*Path.cwd().parents) if (p/"anshp/config.json").is_file())
_A={"__file__":str(_ROOT/"00_setup_and_data_audit.ipynb")}
for _cell in json.loads((_ROOT/"00_setup_and_data_audit.ipynb").read_text(encoding="utf-8"))["cells"]:
    if "er-definitions" in _cell.get("metadata",{}).get("tags",[]):
        exec("".join(_cell["source"]),_A)
_T,_=_A["load_helpers"](_ROOT/"03_qwen_training.ipynb")
_C,_=_A["load_helpers"](_ROOT/"01_data_cleaning.ipynb")

def representative_fit_pairs(audit_dir,reference_count=32):
    audit_dir=Path(audit_dir)
    audit=_A["require_manifest"](audit_dir/"manifest.json","audit",verify_hashes=False)
    con=_A["connect_database"](audit_dir/audit["database"],read_only=True)
    try:
        rows=con.execute("""WITH chosen AS (
            SELECT reference_id FROM splits WHERE split='fit' AND gold_count>0
            ORDER BY md5(reference_id) LIMIT ?),
            one_positive AS (SELECT t.reference_id,min(t.candidate_id) candidate_id
            FROM truth t JOIN chosen USING(reference_id) GROUP BY t.reference_id)
            SELECT q.entity_id,q.business_name,q.business_address,q.country,
                   d.entity_id,d.business_name,d.business_address,d.country
            FROM one_positive t JOIN records q ON q.dataset='train' AND q.entity_id=t.reference_id
            JOIN records d ON d.dataset='train' AND d.entity_id=t.candidate_id ORDER BY q.entity_id""",
            [reference_count]).fetchall()
    finally:con.close()
    if len(rows)<2:raise ValueError("Need at least two fit references for a profile")
    names=("entity_id","business_name","business_address","country")
    positives=[(_C["normalize_record"](dict(zip(names,r[:4]))),
                _C["normalize_record"](dict(zip(names,r[4:])))) for r in rows]
    negatives=[(positives[i][0],positives[(i+1)%len(positives)][1]) for i in range(len(positives))]
    return positives+negatives,[1]*len(positives)+[0]*len(negatives)

def profile_qwen_gpu(audit_dir,config,report_path,batch_sizes=(4,16,32),local_files_only=True):
    """Warm GPU profiling only: forward/backward but zero optimizer updates."""
    import torch
    import gc
    pairs,labels=representative_fit_pairs(audit_dir)
    started=time.perf_counter()
    model,tok,yes_no=_T["load_qwen"](config,train=True,local_files_only=local_files_only)
    load_seconds=time.perf_counter()-started
    device=next(model.parameters()).device
    if device.type!="cuda":raise RuntimeError("This profile requires CUDA")
    model.eval();results=[]
    for batch_size in batch_sizes:
        try:
            batch,_=_T["pair_batch"](tok,pairs[:batch_size],device,config["model"]["max_pair_tokens"])
            with torch.inference_mode():_T["forward_pair_scores"](model,batch,*yes_no)
            torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
            start=time.perf_counter();lengths=[];truncated=0
            with torch.inference_mode():
                for offset in range(0,len(pairs),batch_size):
                    batch,diagnostics=_T["pair_batch"](tok,pairs[offset:offset+batch_size],device,
                                                      config["model"]["max_pair_tokens"])
                    scores=_T["forward_pair_scores"](model,batch,*yes_no)
                    if not bool(torch.isfinite(scores).all()):raise ValueError("Nonfinite profile scores")
                    lengths.extend(batch["attention_mask"].sum(1).cpu().tolist())
                    truncated+=sum(v["truncated"] for fields in diagnostics for v in fields.values())
            torch.cuda.synchronize();seconds=time.perf_counter()-start
            result={"batch_size":batch_size,"pairs":len(pairs),"seconds":seconds,
                    "pairs_per_second":len(pairs)/seconds,
                    "peak_gpu_allocated_bytes":torch.cuda.max_memory_allocated(),
                    "mean_tokens":sum(lengths)/len(lengths),"max_tokens":max(lengths),
                    "truncated_fields":truncated,"success":True}
            results.append(result);print(json.dumps(result),flush=True)
        except torch.OutOfMemoryError:
            results.append({"batch_size":batch_size,"success":False,"reason":"CUDA out of memory"})
            gc.collect();torch.cuda.empty_cache()
            break
    model.train();training_results=[]
    micro=config["training"]["micro_batch_size"]
    for checkpointing in (True,False):
        if checkpointing:model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
        else:model.gradient_checkpointing_disable()
        model.zero_grad(set_to_none=True)
        torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();start=time.perf_counter()
        try:
            for offset in range(0,len(pairs),micro):
                model.zero_grad(set_to_none=True)
                batch,_=_T["pair_batch"](tok,pairs[offset:offset+micro],device,config["model"]["max_pair_tokens"])
                z=_T["forward_pair_scores"](model,batch,*yes_no)
                loss=_T["qwen_independent_pair_loss"](z,labels[offset:offset+micro]).mean()
                if not bool(torch.isfinite(loss)):raise ValueError("Nonfinite profile loss")
                loss.backward()
                if any(not bool(torch.isfinite(p.grad).all()) for p in model.parameters() if p.grad is not None):
                    raise ValueError("Nonfinite adapter gradients")
            torch.cuda.synchronize();seconds=time.perf_counter()-start
            result={"gradient_checkpointing":checkpointing,"pairs_per_second":len(pairs)/seconds,
                    "peak_gpu_allocated_bytes":torch.cuda.max_memory_allocated(),"success":True}
            training_results.append(result);print(json.dumps(result),flush=True)
        except torch.OutOfMemoryError:
            training_results.append({"gradient_checkpointing":checkpointing,"success":False,"reason":"CUDA out of memory"})
            model.zero_grad(set_to_none=True);gc.collect();torch.cuda.empty_cache()
    viable_training=[r for r in training_results if r["success"] and
        r["peak_gpu_allocated_bytes"]<.85*torch.cuda.get_device_properties(0).total_memory]
    if not viable_training:raise RuntimeError("No training mode leaves sufficient GPU memory headroom")
    best_training=max(viable_training,key=lambda r:r["pairs_per_second"])
    successful=[r for r in results if r["success"]]
    if not successful:raise RuntimeError("No inference batch size fit GPU memory")
    best=max(successful,key=lambda r:r["pairs_per_second"])
    audit=_A["require_manifest"](Path(audit_dir)/"manifest.json","audit",verify_hashes=False)
    test_refs=sum(r[3] for r in audit["source_counts"] if r[0]=="test" and r[1]==1)
    report={"revision":config["model"]["revision"],"qwen_helpers":_A["load_helpers"](_ROOT/"03_qwen_training.ipynb")[1],
        "gpu":torch.cuda.get_device_name(0),"gpu_total_bytes":torch.cuda.get_device_properties(0).total_memory,
        "dtype":str(next(model.parameters()).dtype),"load_seconds":load_seconds,
        "inference":results,"recommended_inference_batch_size":best["batch_size"],
        "training":training_results,"recommended_gradient_checkpointing":best_training["gradient_checkpointing"],
        "forward_backward_pairs_per_second":best_training["pairs_per_second"],
        "training_peak_gpu_allocated_bytes":best_training["peak_gpu_allocated_bytes"],
        "trainable_parameters":sum(p.numel() for p in model.parameters() if p.requires_grad),
        "optimizer_steps":0,"validation_accuracy_claim":False,
        "test_hours_upper_candidate_caps_excluding_disk_io":{
            str(cap):test_refs*cap/best["pairs_per_second"]/3600 for cap in (10,20,40)},
        "caveat":"Small fit-only sample; GPU clock, length mix, disk lookup and sustained thermals can change throughput."}
    _A["atomic_json"](report_path,report)
    return report


In [ ]:
RUN_GPU_PROFILE = False
if RUN_GPU_PROFILE:
    CONFIG=json.loads((_ROOT/'config.json').read_text(encoding='utf-8'))
    report=profile_qwen_gpu(_ROOT/'artifacts/audit',CONFIG,_ROOT/'reports/real_qwen_gpu_profile.json')
    print(json.dumps(report,indent=2))
